<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will be able to identify the <b>reaction center</b> using simple bond-delta logic on mapped reactions.
</div>

# S02 · Reaction center by bond deltas (mapped bonds)

**Data:** `data/reactions_mapped.csv`


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: created/deleted bonds
- Hands-on: compute deltas and visualize map numbers


# Theory

A simple reaction-center heuristic is: find mapped bonds that are
- present in products but not reactants (created)
- present in reactants but not products (deleted)

This is strict but easy to teach, and it supports later rule extraction.


# Practical


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw

# Optional: Syn ecosystem (kept optional for Paper 1)
try:
    import synkit  # type: ignore
    HAS_SYNKit = True
except Exception:
    HAS_SYNKit = False

OUT = Path("talktorials/out")
OUT.mkdir(parents=True, exist_ok=True)

import rdkit
import networkx as nx_mod
print("RDKit:", rdkit.__version__)
print("NetworkX:", nx_mod.__version__)
print("SynKit available:", HAS_SYNKit)


In [ ]:
df = pd.read_csv("data/reactions_mapped.csv")
row = df.iloc[0]
react, prod = row.am_rxn_smiles.split(">>")


In [ ]:
def mapped_bonds(m: Chem.Mol) -> set[tuple[int,int,int]]:
    out = set()
    for b in m.GetBonds():
        ai = b.GetBeginAtom().GetAtomMapNum()
        aj = b.GetEndAtom().GetAtomMapNum()
        if ai and aj:
            out.add((min(ai,aj), max(ai,aj), int(b.GetBondTypeAsDouble())))
    return out

mR = Chem.MolFromSmiles(react)
mP = Chem.MolFromSmiles(prod)
bR = mapped_bonds(mR)
bP = mapped_bonds(mP)

created = bP - bR
deleted = bR - bP
print("Created:", created)
print("Deleted:", deleted)


In [ ]:
def draw_mapnums(smiles: str, size=(450,250)):
    m = Chem.MolFromSmiles(smiles)
    for a in m.GetAtoms():
        amap = a.GetAtomMapNum()
        if amap:
            a.SetProp("atomNote", str(amap))
    return Draw.MolToImage(m, size=size)

display(draw_mapnums(react))
display(draw_mapnums(prod))


# Discussion

- Bond deltas capture many transformations, but they miss:
  - changes in bond order (single→double)
  - stereochemistry changes
  - changes involving unmapped atoms
Use this as a first approximation, then refine later if needed.


# Quiz
1. How would you include bond order changes in the delta definition?
2. Why is atom mapping essential to compare bonds?
3. What transformations might bond-delta miss completely?


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
